# Householder-RoPE Colab Scaling Harness

## What this is
This notebook is a Colab harness for scaling Householder-RoPE on a realistic language-modeling workload. It drives the reusable runner in `scripts/run_real_data_scale_harness.py`, targets a single GPU by default, and switches to Flax on TPU when a TPU runtime is detected.

## Why it matters
The repository already verifies the Householder transport algebra and synthetic attention-block benchmarks. This notebook moves the test into a more realistic regime: tokenized `wikitext-103-raw-v1`, causal next-token prediction, larger model settings, and saved artifacts that are easy to compare across accelerators while exposing live optimization and RoPE-health telemetry.

## How to run / use it
1. Open the notebook in Colab and choose a GPU or TPU runtime.
2. Run the repo bootstrap cell. It clones `https://github.com/fyremael/HH.git` when the repo is not already present.
3. Adjust `COLAB_CONFIG` for your accelerator budget. The default profile is a 40GB A100-oriented capacity-max run with a 4096-token context that is meant to push VRAM usage much harder than the earlier profiles. The notebook automatically backs off the batch size all the way down to 1 if CUDA OOM occurs, and it increases gradient accumulation so we stay near the original effective batch size instead of failing outright.
4. Run the harness cell. The script emits ongoing stepwise metrics, per-variant JSONL and CSV histories, and multiple PNG diagnostics panels into `artifacts/`, and the notebook streams the child process logs line by line into the Colab cell output.
5. The harness cell will retry smaller batch sizes automatically on CUDA OOM, will raise gradient accumulation when needed, and will report the effective batch size and accumulation that succeeded.
6. If even the backoff path fails, switch `ACTIVE_PROFILE` to `capacity_limit`, then to `geometry_signal` if needed.
7. Run the inspection cell to display the summary table, the compact loss-throughput-memory tradeoff view, the environment report, recent probe diagnostics, and the saved plots.

## Validation plan
Validation here means three concrete checks: the package imports cleanly inside Colab, the realistic-data training loop completes without NaNs, the A100 profile uses materially more memory than the earlier small runs, and the ablation summary reports comparable losses and throughput for the requested reflector sweep while the diagnostics remain numerically well-behaved.

## Known failure modes
TPU runtimes may need one restart after installing `jax[tpu]`. `wikitext-103-raw-v1` is much larger than the smoke datasets used locally, so smaller GPUs may need lower `train_text_limit`, `seq_len`, or `batch_size`. If Colab loses its working directory, rerun the repo bootstrap cell before reinstalling dependencies. Dense diagnostics are intentionally limited by `diagnostic_token_limit` to keep the probe pass affordable at long context.

## Next steps
The next natural extensions are repeated-seed comparisons, throughput-vs-perplexity reports, deeper stacks with checkpointing, and a merge report that compares these realistic-data runs with the synthetic backend benchmarks already stored in `artifacts/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def run_command(*command: str) -> None:
    print("+", " ".join(command))
    subprocess.run(list(command), check=True)

REPO_URL = "https://github.com/fyremael/HH.git"
REPO_CANDIDATES = [Path.cwd(), Path("/content/HH"), Path("/content/drive/MyDrive/HH")]
REPO_PATH = next((path for path in REPO_CANDIDATES if (path / "src" / "householder_rope").exists()), None)
if REPO_PATH is None:
    target = Path("/content/HH")
    if not target.exists():
        run_command("git", "clone", REPO_URL, str(target))
    REPO_PATH = target

ARTIFACT_DIR = REPO_PATH / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

run_command(sys.executable, "-m", "pip", "install", "-q", "uv")
run_command(
    "uv",
    "pip",
    "install",
    "--system",
    "datasets>=3.2.0",
    "transformers>=4.49.0",
    "sentencepiece>=0.2.0",
    "matplotlib>=3.9.0",
    "pandas>=2.2.0",
    "tqdm>=4.66.0",
)
if "COLAB_TPU_ADDR" in os.environ:
    run_command("uv", "pip", "install", "--system", "flax==0.12.5", "optax==0.2.6")
    try:
        import jax
        have_tpu = any(device.platform == "tpu" for device in jax.devices())
    except Exception:
        have_tpu = False
    if not have_tpu:
        run_command(
            "uv",
            "pip",
            "install",
            "--system",
            "jax[tpu]",
            "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        )
        raise SystemExit("Installed jax[tpu]. Restart the runtime once, then rerun the notebook from the top.")
run_command("uv", "pip", "install", "--system", "--no-deps", "-e", str(REPO_PATH))

print(f"Repo path: {REPO_PATH}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
import json
import os

A100_PRESETS = {
    "fast_sanity": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 12000,
        "eval_text_limit": 1500,
        "seq_len": 256,
        "batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 1,
        "train_steps": 40,
        "eval_every": 10,
        "eval_batches": 6,
        "log_every": 5,
        "diagnostics_every": 10,
        "diagnostic_token_limit": 96,
        "num_layers": 2,
        "embed_dim": 512,
        "num_heads": 8,
        "mlp_ratio": 4.0,
        "learning_rate": 3.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8],
        "output_stem": "colab_a100_fast_sanity",
    },
    "serious_comparison": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 60000,
        "eval_text_limit": 8000,
        "seq_len": 1024,
        "batch_size": 6,
        "eval_batch_size": 6,
        "gradient_accumulation_steps": 1,
        "train_steps": 300,
        "eval_every": 20,
        "eval_batches": 8,
        "log_every": 5,
        "diagnostics_every": 10,
        "diagnostic_token_limit": 192,
        "num_layers": 6,
        "embed_dim": 1024,
        "num_heads": 16,
        "mlp_ratio": 4.0,
        "learning_rate": 2.5e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8, 16],
        "output_stem": "colab_a100_40gb_default",
    },
    "geometry_signal": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 80000,
        "eval_text_limit": 10000,
        "seq_len": 2048,
        "batch_size": 3,
        "eval_batch_size": 3,
        "gradient_accumulation_steps": 1,
        "train_steps": 600,
        "eval_every": 25,
        "eval_batches": 8,
        "log_every": 10,
        "diagnostics_every": 25,
        "diagnostic_token_limit": 256,
        "num_layers": 6,
        "embed_dim": 1024,
        "num_heads": 16,
        "mlp_ratio": 4.0,
        "learning_rate": 2.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 16, 32],
        "output_stem": "colab_a100_geometry_signal",
    },
    "capacity_limit": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 120000,
        "eval_text_limit": 12000,
        "seq_len": 4096,
        "batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 1,
        "train_steps": 400,
        "eval_every": 25,
        "eval_batches": 6,
        "log_every": 5,
        "diagnostics_every": 25,
        "diagnostic_token_limit": 256,
        "num_layers": 12,
        "embed_dim": 1536,
        "num_heads": 24,
        "mlp_ratio": 4.0,
        "learning_rate": 1.5e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 16, 32],
        "output_stem": "colab_a100_capacity_limit",
    },
    "capacity_max": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 200000,
        "eval_text_limit": 16000,
        "seq_len": 4096,
        "batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 1,
        "train_steps": 300,
        "eval_every": 25,
        "eval_batches": 4,
        "log_every": 5,
        "diagnostics_every": 25,
        "diagnostic_token_limit": 256,
        "num_layers": 24,
        "embed_dim": 2048,
        "num_heads": 32,
        "mlp_ratio": 4.0,
        "learning_rate": 1.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 16, 32],
        "output_stem": "colab_a100_capacity_max",
    },
    "long_context_stress": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 50000,
        "eval_text_limit": 6000,
        "seq_len": 1024,
        "batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 1,
        "train_steps": 120,
        "eval_every": 20,
        "eval_batches": 6,
        "log_every": 10,
        "diagnostics_every": 20,
        "diagnostic_token_limit": 192,
        "num_layers": 4,
        "embed_dim": 768,
        "num_heads": 12,
        "mlp_ratio": 4.0,
        "learning_rate": 2.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8, 16],
        "output_stem": "colab_a100_long_context_stress",
    },
}

ACTIVE_PROFILE = "capacity_max"
COLAB_CONFIG = dict(A100_PRESETS[ACTIVE_PROFILE])

BACKEND = COLAB_CONFIG["backend"]
if BACKEND == "auto":
    BACKEND = "flax" if "COLAB_TPU_ADDR" in os.environ else "torch"

print("Available profiles:", ", ".join(A100_PRESETS))
print("Selected profile:", ACTIVE_PROFILE)
print("Selected backend:", BACKEND)
print("TPU runtime hint:", "COLAB_TPU_ADDR" in os.environ)
print("If this profile OOMs, switch ACTIVE_PROFILE to 'capacity_limit', then 'geometry_signal'.")
print(json.dumps(COLAB_CONFIG, indent=2))


In [ ]:
import math
import os
import subprocess
import sys

def build_command(run_config: dict[str, object], output_stem: str) -> list[str]:
    command = [
        sys.executable,
        "-u",
        str(REPO_PATH / "scripts" / "run_real_data_scale_harness.py"),
        "--backend",
        BACKEND,
        "--dataset-name",
        str(run_config["dataset_name"]),
        "--dataset-config",
        str(run_config["dataset_config"]),
        "--tokenizer-name",
        str(run_config["tokenizer_name"]),
        "--train-text-limit",
        str(run_config["train_text_limit"]),
        "--eval-text-limit",
        str(run_config["eval_text_limit"]),
        "--seq-len",
        str(run_config["seq_len"]),
        "--batch-size",
        str(run_config["batch_size"]),
        "--eval-batch-size",
        str(run_config["eval_batch_size"]),
        "--gradient-accumulation-steps",
        str(run_config["gradient_accumulation_steps"]),
        "--train-steps",
        str(run_config["train_steps"]),
        "--eval-every",
        str(run_config["eval_every"]),
        "--eval-batches",
        str(run_config["eval_batches"]),
        "--log-every",
        str(run_config["log_every"]),
        "--diagnostics-every",
        str(run_config["diagnostics_every"]),
        "--diagnostic-token-limit",
        str(run_config["diagnostic_token_limit"]),
        "--num-layers",
        str(run_config["num_layers"]),
        "--embed-dim",
        str(run_config["embed_dim"]),
        "--num-heads",
        str(run_config["num_heads"]),
        "--mlp-ratio",
        str(run_config["mlp_ratio"]),
        "--learning-rate",
        str(run_config["learning_rate"]),
        "--weight-decay",
        str(run_config["weight_decay"]),
        "--seed",
        str(run_config["seed"]),
        "--householder-init",
        str(run_config["householder_init"]),
        "--reflector-sweep",
        *[str(value) for value in run_config["reflector_sweep"]],
        "--output-stem",
        output_stem,
        "--log-level",
        "INFO",
    ]
    if run_config["use_compile"]:
        command.append("--use-compile")
    else:
        command.append("--no-use-compile")
    if run_config["use_bf16"]:
        command.append("--use-bf16")
    else:
        command.append("--no-use-bf16")
    return command

def run_command_stream(command: list[str], env: dict[str, str]) -> tuple[int, str]:
    process = subprocess.Popen(
        command,
        cwd=REPO_PATH,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    collected_lines: list[str] = []
    for line in process.stdout:
        collected_lines.append(line)
        print(line, end="", flush=True)
    return process.wait(), "".join(collected_lines)

run_config = dict(COLAB_CONFIG)
attempted_batches = []
RUN_COLAB_CONFIG = None
RUN_OUTPUT_STEM = None

base_batch = int(COLAB_CONFIG["batch_size"])
base_eval_batch = int(COLAB_CONFIG["eval_batch_size"])
base_grad_accum = int(COLAB_CONFIG["gradient_accumulation_steps"])
base_effective_batch = base_batch * base_grad_accum
candidate_batches = list(range(base_batch, 0, -1))
env = dict(
    os.environ,
    PYTHONUNBUFFERED="1",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)
print("Live harness logs will stream below.", flush=True)
print("Batch-size backoff ladder:", candidate_batches, flush=True)
print("Base effective batch:", base_effective_batch, flush=True)

for attempt_batch in candidate_batches:
    attempted_batches.append(attempt_batch)
    run_config["batch_size"] = attempt_batch
    run_config["eval_batch_size"] = min(base_eval_batch, attempt_batch)
    run_config["gradient_accumulation_steps"] = max(
        base_grad_accum,
        max(1, base_effective_batch // attempt_batch),
    )
    output_stem = (
        f"{COLAB_CONFIG['output_stem']}_bs{attempt_batch}"
        f"_ga{run_config['gradient_accumulation_steps']}"
    )
    command = build_command(run_config, output_stem)
    print(
        f"Attempting batch_size={attempt_batch}, "
        f"gradient_accumulation_steps={run_config['gradient_accumulation_steps']}",
        flush=True,
    )
    print("+", " ".join(command), flush=True)
    return_code, combined_output = run_command_stream(command, env)
    if return_code == 0:
        RUN_COLAB_CONFIG = dict(run_config)
        RUN_COLAB_CONFIG["output_stem"] = output_stem
        RUN_OUTPUT_STEM = output_stem
        print(
            f"Run succeeded with batch_size={attempt_batch} and "
            f"gradient_accumulation_steps={run_config['gradient_accumulation_steps']}.",
            flush=True,
        )
        break
    oom_markers = ("torch.OutOfMemoryError", "CUDA out of memory")
    if BACKEND == "torch" and any(marker in combined_output for marker in oom_markers):
        print(
            f"CUDA OOM at batch_size={attempt_batch}. Retrying with a smaller batch.",
            flush=True,
        )
        continue
    raise subprocess.CalledProcessError(return_code, command)
else:
    raise RuntimeError(f"All attempted batch sizes failed: {attempted_batches}")


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

effective_config = dict(COLAB_CONFIG) if RUN_COLAB_CONFIG is None else dict(RUN_COLAB_CONFIG)
stem = effective_config["output_stem"]
metrics_path = ARTIFACT_DIR / f"{stem}_metrics.json"
summary_path = ARTIFACT_DIR / f"{stem}_summary.csv"
loss_curve_path = ARTIFACT_DIR / f"{stem}_loss_curves.png"
throughput_path = ARTIFACT_DIR / f"{stem}_throughput.png"
dynamics_path = ARTIFACT_DIR / f"{stem}_training_dynamics.png"
component_path = ARTIFACT_DIR / f"{stem}_component_diagnostics.png"
rope_path = ARTIFACT_DIR / f"{stem}_rope_diagnostics.png"

payload = json.loads(metrics_path.read_text(encoding="utf-8"))
summary_df = pd.read_csv(summary_path)
key_columns = [
    "variant",
    "final_eval_loss",
    "mean_tokens_per_second",
    "peak_memory_gb",
    "latest_probe_loss",
    "latest_rope_identity_deviation_mean",
]
tradeoff_df = summary_df[key_columns].copy()
print("Selected profile:", ACTIVE_PROFILE)
print("Attempted batches:", attempted_batches)
print("Effective run stem:", stem)
print("Effective batch size:", effective_config["batch_size"])
print("Effective gradient accumulation:", effective_config["gradient_accumulation_steps"])
print(
    "Effective tokens per optimizer step:",
    effective_config["batch_size"] * effective_config["gradient_accumulation_steps"] * effective_config["seq_len"],
)
display(tradeoff_df)
display(summary_df)
if not tradeoff_df.empty and "standard_rope" in set(tradeoff_df["variant"]):
    baseline_row = tradeoff_df.loc[tradeoff_df["variant"] == "standard_rope"].iloc[0]
    comparison_df = tradeoff_df.copy()
    comparison_df["eval_loss_delta_vs_standard"] = comparison_df["final_eval_loss"] - baseline_row["final_eval_loss"]
    comparison_df["throughput_ratio_vs_standard"] = comparison_df["mean_tokens_per_second"] / baseline_row["mean_tokens_per_second"]
    display(comparison_df)
    peak_memory = float(tradeoff_df["peak_memory_gb"].max())
    if peak_memory < 25.0:
        print(f"Peak memory still looks low for a 40GB A100: {peak_memory:.2f} GB. Consider increasing batch size again.")
print(json.dumps(payload["environment"], indent=2))
print(json.dumps(payload["dataset_summary"], indent=2))
for result in payload["results"]:
    print(result["variant"], json.dumps(result.get("latest_probe_summary"), indent=2))
    print("history_jsonl_path:", result["history_jsonl_path"])
    print("history_csv_path:", result["history_csv_path"])
display(Image(filename=str(loss_curve_path)))
display(Image(filename=str(throughput_path)))
display(Image(filename=str(dynamics_path)))
display(Image(filename=str(component_path)))
display(Image(filename=str(rope_path)))
